# Project: Tomographic reconstruction with Huber regularization

We want to solve
$$ \min_x \frac{1}{2} \| A(x) - y \|_2^2 + \lambda H_\epsilon(\nabla x) $$
with steepest descent. The method is first tested in 2D on the Shepp-Logan phantom.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import odl
from odl.applications import tomo

## 2D setup

In [ ]:
reco_space_2d = odl.uniform_discr([-112, -112], [112, 112], (128, 128), dtype='float32')
geometry_2d = tomo.cone_beam_geometry(
    space=reco_space_2d,
    src_radius=780.0,
    det_radius=220.0,
    num_angles=128,
    short_scan=True,
)
ray_trafo_2d = tomo.RayTransform(vol_space=reco_space_2d, geometry=geometry_2d)
data_space_2d = ray_trafo_2d.range

phantom = odl.phantom.shepp_logan(reco_space_2d, modified=False)
exact_data = ray_trafo_2d(phantom)

In [ ]:
# absolute path, so that it does not change when the working directory changes
fig_dir = os.path.abspath('figures')
os.makedirs(fig_dir, exist_ok=True)


def save_figure(name):
    plt.savefig(os.path.join(fig_dir, f'{name}.svg'), bbox_inches='tight')


def show_images(images, titles, clim=(0.98, 1.06), filename=None):
    # The default window shows the inner ellipses, the skull (value 2) is saturated
    vmin, vmax = clim if clim is not None else (None, None)
    fig, axes = plt.subplots(1, len(images), figsize=(4 * len(images), 4))
    for ax, image, title in zip(np.atleast_1d(axes), images, titles):
        ax.imshow(image.asarray().T, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    if filename is not None:
        save_figure(filename)
    plt.show()


def add_noise(data, relative_noise, seed=0):
    noise = odl.phantom.white_noise(data.space, seed=seed)
    noise = float(relative_noise * data.norm() / noise.norm()) * noise
    return data + noise, noise.norm()


def relative_error(x, x_true):
    return (x - x_true).norm() / x_true.norm()

In [ ]:
show_images([phantom], ['Phantom (full range)'], clim=(0, 2), filename='phantom_full')
show_images([phantom], ['Phantom (narrow window)'], filename='phantom_window')

In [ ]:
noisy_data, delta = add_noise(exact_data, 0.01)
print(f'Noise level delta = {delta:.2f}, relative noise = {delta / exact_data.norm():.3f}')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(exact_data.asarray(), aspect='auto')
axes[0].set_title('Exact data')
axes[1].imshow(noisy_data.asarray(), aspect='auto')
axes[1].set_title('Noisy data (1% noise)')
save_figure('data_2d')
plt.show()

## Solver

The gradient of the objective is Lipschitz with constant $L = \|A\|^2 + \lambda \|\nabla\|^2 / \epsilon$,
so steepest descent with constant step $t < 2/L$ converges. We use $t = 1.9/L$.
The operator norms are only estimated once since this is slow in 3D.

In [ ]:
ray_trafo_norm_2d = float(ray_trafo_2d.norm(estimate=True))
grad_norm_2d = float(odl.Gradient(reco_space_2d).norm(estimate=True))
op_norms_2d = (ray_trafo_norm_2d, grad_norm_2d)
print(f'||A|| = {ray_trafo_norm_2d:.2f}, ||grad|| = {grad_norm_2d:.2f}')

In [ ]:
def huber_reconstruction(ray_trafo, data, lam, eps, niter, op_norms, x_true=None, x0=None):
    lam = float(lam)
    grad = odl.Gradient(ray_trafo.domain)
    data_fit = 0.5 * odl.functionals.L2NormSquared(ray_trafo.range).translated(data) * ray_trafo
    regularizer = lam * odl.functionals.Huber(grad.range, eps) * grad
    objective = data_fit + regularizer

    ray_trafo_norm, grad_norm = op_norms
    L = ray_trafo_norm**2 + lam * grad_norm**2 / eps
    step = 1.9 / L

    errors = []
    def callback(x):
        if x_true is not None:
            errors.append(relative_error(x, x_true))

    x = ray_trafo.domain.zero() if x0 is None else x0.copy()
    odl.solvers.steepest_descent(objective, x, line_search=step, maxiter=niter, callback=callback)
    return x, errors

### First test

In [ ]:
niter = 3000

x, errors = huber_reconstruction(ray_trafo_2d, noisy_data, lam=1.0, eps=0.01, niter=niter,
                                 op_norms=op_norms_2d, x_true=phantom)

show_images([phantom, x], ['Phantom', f'Huber, error = {errors[-1]:.3f}'], filename='first_test_reco')

plt.figure()
plt.plot(errors)
plt.xlabel('Iteration')
plt.ylabel('Relative error')
plt.yscale('log')
plt.grid(True)
plt.title(r'Relative error, $\lambda = 1$, $\epsilon = 0.01$')
save_figure('first_test_error')
plt.show()

## Choice of $\epsilon$

$\lambda$ is fixed and we compare different smoothing parameters $\epsilon$.

In [ ]:
eps_values = [1e-3, 1e-2, 1e-1, 1.0]

eps_recos = []
plt.figure()
for eps in eps_values:
    x, errors = huber_reconstruction(ray_trafo_2d, noisy_data, lam=1.0, eps=eps, niter=niter,
                                     op_norms=op_norms_2d, x_true=phantom)
    eps_recos.append(x)
    plt.plot(errors, label=rf'$\epsilon = {eps}$')
    print(f'eps = {eps}: final error = {errors[-1]:.4f}')

plt.xlabel('Iteration')
plt.ylabel('Relative error')
plt.yscale('log')
plt.grid(True)
plt.title(r'Relative error for different $\epsilon$ ($\lambda = 1$)')
plt.legend()
save_figure('eps_error')
plt.show()

show_images(eps_recos, [rf'$\epsilon = {eps}$' for eps in eps_values], filename='eps_recos')

In [ ]:
eps = 1e-3  # chosen from the experiment above

## Choice of $\lambda$: Morozov's discrepancy principle

For a decreasing sequence $\lambda_1 > \lambda_2 > \dots$ we pick the first $\lambda_i$ such that
$\| A x_{\lambda_i} - y \| \le \tau \delta$ with $\tau > 1$.

In [ ]:
tau = 1.1
lam_values = np.logspace(1, -2, 10)

residuals = []
final_errors = []
recos = []
for lam in lam_values:
    x, errors = huber_reconstruction(ray_trafo_2d, noisy_data, lam, eps, niter, op_norms_2d, x_true=phantom)
    recos.append(x)
    residuals.append((ray_trafo_2d(x) - noisy_data).norm())
    final_errors.append(errors[-1])
    print(f'lambda = {lam:.4f}: residual / delta = {residuals[-1] / delta:.3f}, error = {errors[-1]:.4f}')

# first (largest) lambda that satisfies the discrepancy principle
for i_morozov, res in enumerate(residuals):
    if res <= tau * delta:
        break
else:
    print('No lambda satisfies the discrepancy principle, using the smallest one')
lam_morozov = lam_values[i_morozov]
print(f'Morozov: lambda = {lam_morozov:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].loglog(lam_values, residuals, 'o-', label='residual')
axes[0].axhline(tau * delta, color='r', linestyle='--', label=r'$\tau \delta$')
axes[0].axvline(lam_morozov, color='k', linestyle=':', label=r'Morozov $\lambda$')
axes[0].set_xlabel(r'$\lambda$')
axes[0].set_ylabel(r'$\|Ax - y\|$')
axes[0].grid(True)
axes[0].legend()

axes[1].semilogx(lam_values, final_errors, 'o-')
axes[1].axvline(lam_morozov, color='k', linestyle=':', label=r'Morozov $\lambda$')
axes[1].set_xlabel(r'$\lambda$')
axes[1].set_ylabel('Relative error')
axes[1].grid(True)
axes[1].legend()
save_figure('morozov_2d')
plt.show()

x_morozov = recos[i_morozov]
show_images([phantom, x_morozov], ['Phantom', rf'$\lambda = {lam_morozov:.3f}$, error = {final_errors[i_morozov]:.3f}'],
            filename='morozov_2d_reco')

## Error vs iterations for different $\lambda$ and noise levels

In [ ]:
noise_levels = [0.01, 0.03, 0.05]
lam_test = [0.1, 1.0, 10.0]

fig, axes = plt.subplots(1, len(noise_levels), figsize=(15, 4), sharey=True)
for ax, noise in zip(axes, noise_levels):
    noisy, _ = add_noise(exact_data, noise)
    for lam in lam_test:
        _, errors = huber_reconstruction(ray_trafo_2d, noisy, lam, eps, niter, op_norms_2d, x_true=phantom)
        ax.plot(errors, label=rf'$\lambda = {lam}$')
    ax.set_title(f'{noise * 100:.0f}% noise')
    ax.set_xlabel('Iteration')
    ax.set_yscale('log')
    ax.grid(True)
    ax.legend()
axes[0].set_ylabel('Relative error')
save_figure('error_vs_iterations')
plt.show()

## Filtered back-projection

The short scan covers $180°$ plus the fan angle, so some lines are measured twice and others once.
FBP assumes every line is counted equally, which gives a shading over the image. Parker weighting fixes
this by weighting each ray so that the weights of two rays measuring the same line sum to 1.

In [ ]:
def parker_weights(ray_trafo):
    weights = tomo.parker_weighting(ray_trafo)
    if ray_trafo.domain.ndim == 2:
        # In 2D, ODL's Parker weights are mirrored along the detector axis, so we flip them back
        weights = ray_trafo.range.element(np.flip(weights.asarray(), axis=1).copy())
    return weights


parker = parker_weights(ray_trafo_2d)

### Check on noise-free data: short scan vs full 360° scan

In [ ]:
fbp_short = tomo.fbp_op(ray_trafo_2d)

geometry_full = tomo.cone_beam_geometry(
    space=reco_space_2d,
    src_radius=780.0,
    det_radius=220.0,
    num_angles=128,
    short_scan=False,
)
ray_trafo_full = tomo.RayTransform(vol_space=reco_space_2d, geometry=geometry_full)
x_full = tomo.fbp_op(ray_trafo_full)(ray_trafo_full(phantom))

images = [
    fbp_short(exact_data),
    fbp_short(tomo.parker_weighting(ray_trafo_2d) * exact_data),
    fbp_short(parker * exact_data),
    x_full,
]
titles = ['Short scan, no weighting', 'Short scan, ODL Parker', 'Short scan, flipped Parker', 'Full 360° scan']
show_images(images, [f'{t}\nerror = {relative_error(x, phantom):.3f}' for t, x in zip(titles, images)],
            filename='parker_check')

### Filter types and frequency scaling (noisy data, with Parker weighting)

In [ ]:
scalings = [1.0, 0.5, 0.25]

for filter_type in ['Ram-Lak', 'Hann']:
    images, titles = [], []
    for scaling in scalings:
        fbp = tomo.fbp_op(ray_trafo_2d, filter_type=filter_type, frequency_scaling=scaling)
        x_fbp = fbp(parker * noisy_data)
        images.append(x_fbp)
        titles.append(f'{filter_type}, scaling = {scaling}\nerror = {relative_error(x_fbp, phantom):.3f}')
    show_images(images, titles, filename=f'fbp_2d_{filter_type}')

In [ ]:
x_fbp = tomo.fbp_op(ray_trafo_2d, filter_type='Hann', frequency_scaling=0.5)(parker * noisy_data)
show_images([phantom, x_fbp, x_morozov],
            ['Phantom',
             f'FBP (Hann, 0.5), error = {relative_error(x_fbp, phantom):.3f}',
             f'Huber, error = {relative_error(x_morozov, phantom):.3f}'],
            filename='comparison_2d')

# 3D reconstruction

The noise in the 3D data has a relative norm of about 1/100, so we use $\delta = 0.01 \|y\|$.
The parameter $\epsilon$ is taken from the 2D tests.

In [ ]:
import time
from odl.contrib.tomo import elekta_icon_geometry

reco_space = odl.uniform_discr([-112, -112, 0], [112, 112, 224], (128, 128, 128), dtype='float32')
geometry = elekta_icon_geometry(num_angles=128)
ray_trafo = tomo.RayTransform(vol_space=reco_space, geometry=geometry)
data_space = ray_trafo.range

data_path = '/home/jupyter-miniproject8/InverseProblems-Labs/noisy_sinogram.npy'
data = data_space.element(np.load(data_path))

delta_3d = 0.01 * data.norm()
print(f'Data shape: {data_space.shape}, delta = {delta_3d:.2f}')

In [ ]:
def show_slices(x, title, clim=None, filename=None):
    # axial, coronal and sagittal slices through the middle of the volume
    volume = x.asarray()
    nx, ny, nz = volume.shape
    slices = [volume[:, :, nz // 2], volume[:, ny // 2, :], volume[nx // 2, :, :]]
    vmin, vmax = clim if clim is not None else (None, None)
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, image, name in zip(axes, slices, ['Axial', 'Coronal', 'Sagittal']):
        ax.imshow(image.T, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
        ax.set_title(name)
        ax.axis('off')
    fig.suptitle(title)
    plt.tight_layout()
    if filename is not None:
        save_figure(filename)
    plt.show()


def axial_slice(x):
    # the 2D space is exactly an axial slice of the 3D space
    volume = x.asarray()
    return reco_space_2d.element(volume[:, :, volume.shape[2] // 2])

## Filtered back-projection in 3D

In [ ]:
parker_3d = parker_weights(ray_trafo)
fbp = tomo.fbp_op(ray_trafo)

show_slices(fbp(data), 'FBP without Parker weighting', filename='fbp_3d_no_parker')

x_fbp_3d = fbp(parker_3d * data)
show_slices(x_fbp_3d, 'FBP with Parker weighting', filename='fbp_3d_parker')
print(f'min = {odl.min(x_fbp_3d):.3f}, max = {odl.max(x_fbp_3d):.3f}')

In [ ]:
# display window centred on the brain values (central block of the FBP volume)
brain = x_fbp_3d.asarray()[48:80, 48:80, 60:68]
print(f'brain: mean = {brain.mean():.3f}, std = {brain.std():.3f}')
clim_3d = (brain.mean() - 3 * brain.std(), brain.mean() + 3 * brain.std())

In [ ]:
for filter_type in ['Ram-Lak', 'Hann']:
    images = []
    for scaling in scalings:
        fbp = tomo.fbp_op(ray_trafo, filter_type=filter_type, frequency_scaling=scaling)
        images.append(axial_slice(fbp(parker_3d * data)))
    show_images(images, [f'{filter_type}, scaling = {scaling}' for scaling in scalings], clim=clim_3d,
                filename=f'fbp_3d_{filter_type}')

## Huber reconstruction in 3D

In [ ]:
t = time.time()
ray_trafo_norm = float(ray_trafo.norm(estimate=True))
grad_norm = float(odl.Gradient(reco_space).norm(estimate=True))
op_norms_3d = (ray_trafo_norm, grad_norm)
print(f'||A|| = {ray_trafo_norm:.2f}, ||grad|| = {grad_norm:.2f} ({time.time() - t:.0f} s)')

### Short test to measure the time per iteration

In [ ]:
t = time.time()
x, _ = huber_reconstruction(ray_trafo, data, lam=1.0, eps=eps, niter=10, op_norms=op_norms_3d)
print(f'{(time.time() - t) / 10:.2f} s per iteration')
print(f'residual / delta = {(ray_trafo(x) - data).norm() / delta_3d:.3f}')
show_slices(x, r'10 iterations, $\lambda = 1$', clim=clim_3d, filename='huber_3d_test')

### Choice of $\lambda$ with Morozov's discrepancy principle

With $\delta = 0.01 \|y\|$ the residual never gets below about $1.34\delta$, even with almost no regularization,
so the data contains errors that the model cannot explain. We therefore estimate the total data error $\delta_{\text{eff}}$
as the residual of the almost unregularized reconstruction, and pick the first $\lambda$ with $\| A x - y \| \le \tau \delta_{\text{eff}}$.
Each $\lambda$ starts from the reconstruction of the previous one to save iterations.

In [ ]:
niter_3d = 200
lam_values_3d = [1.0, 0.3, 0.1, 0.03, 0.01, 0.003, 0.001]

recos_3d = []
ratios_3d = []
x = None
for lam in lam_values_3d:
    t = time.time()
    x, _ = huber_reconstruction(ray_trafo, data, lam, eps, niter_3d, op_norms_3d, x0=x)
    recos_3d.append(x)
    ratios_3d.append((ray_trafo(x) - data).norm() / delta_3d)
    print(f'lambda = {lam}: residual / delta = {ratios_3d[-1]:.3f} ({time.time() - t:.0f} s)')

# residual of the almost unregularized reconstruction
delta_eff = min(ratios_3d) * delta_3d
print(f'delta_eff = {delta_eff / delta_3d:.3f} delta')

# first (largest) lambda that satisfies the discrepancy principle with delta_eff
for i_morozov_3d, ratio in enumerate(ratios_3d):
    if ratio * delta_3d <= tau * delta_eff:
        break
lam_morozov_3d = lam_values_3d[i_morozov_3d]
print(f'Morozov: lambda = {lam_morozov_3d}')

In [ ]:
plt.figure()
plt.semilogx(lam_values_3d, ratios_3d, 'o-', label='residual')
plt.axhline(tau, color='r', linestyle='--', label=r'$\tau \delta$')
plt.axhline(tau * delta_eff / delta_3d, color='g', linestyle='--', label=r'$\tau \delta_{\mathrm{eff}}$')
plt.axvline(lam_morozov_3d, color='k', linestyle=':', label=r'Morozov $\lambda$')
plt.xlabel(r'$\lambda$')
plt.ylabel(r'$\|Ax - y\| / \delta$')
plt.grid(True)
plt.legend()
save_figure('morozov_3d')
plt.show()

show_images([axial_slice(x) for x in recos_3d], [rf'$\lambda = {lam}$' for lam in lam_values_3d], clim=clim_3d,
            filename='lambda_3d')

### Check of the largest $\lambda$

$\lambda = 1$ is the first value of the sweep, so it starts from zero and has the smallest step size.
We continue it to see if its residual goes below $\tau \delta_{\text{eff}}$ once it has converged.

In [ ]:
x_check = recos_3d[0]
for k in range(2, 6):
    x_check, _ = huber_reconstruction(ray_trafo, data, lam_values_3d[0], eps, 200, op_norms_3d, x0=x_check)
    ratio = (ray_trafo(x_check) - data).norm() / delta_3d
    print(f'{200 * k} iterations: residual / delta = {ratio:.3f} (threshold = {tau * delta_eff / delta_3d:.3f})')

show_images([axial_slice(recos_3d[0]), axial_slice(x_check)],
            [rf'$\lambda = {lam_values_3d[0]}$, 200 iterations', rf'$\lambda = {lam_values_3d[0]}$, 1000 iterations'],
            clim=clim_3d, filename='lambda_check_3d')

### Evolution with the number of iterations

In [ ]:
checkpoints = [10, 50, 200, 500, 1000]

# steepest descent with a constant step has no memory, so we can continue from the last iterate
iterates = []
x = None
done = 0
for k in checkpoints:
    x, _ = huber_reconstruction(ray_trafo, data, lam_morozov_3d, eps, k - done, op_norms_3d, x0=x)
    done = k
    iterates.append(x)
    print(f'{k} iterations: residual / delta = {(ray_trafo(x) - data).norm() / delta_3d:.3f}')

show_images([axial_slice(x) for x in iterates], [f'{k} iterations' for k in checkpoints], clim=clim_3d,
            filename='iterations_3d')

In [ ]:
x_huber_3d = iterates[-1]
x_hann_3d = tomo.fbp_op(ray_trafo, filter_type='Hann', frequency_scaling=0.25)(parker_3d * data)

show_slices(x_fbp_3d, 'FBP (Ram-Lak, scaling = 1.0)', clim=clim_3d, filename='final_fbp_ramlak')
show_slices(x_hann_3d, 'FBP (Hann, scaling = 0.25)', clim=clim_3d, filename='final_fbp_hann')
show_slices(x_huber_3d, rf'Huber, $\lambda = {lam_morozov_3d}$, {checkpoints[-1]} iterations', clim=clim_3d,
            filename='final_huber')